# 🏷️ Multi-Label Classification: Asymmetric Loss (ASL)

### 📦 Installation
```bash
pip install -e .
```

### 🎯 Overview
When dealing with items that belong to multiple overlapping categories (e.g., an image of an urban sunset), traditional multiclass algorithms fail.

`FastMultiLabelKMeansClassifier` natively handles this by building **independent sub-topologies** for every tag. 
*   **Asymmetric Loss (ASL):** Dynamically scales down the gradients of dominant 'Negative' classes, preventing gradient starvation in highly sparse label matrices.
*   **Gap Strategy:** During prediction, `strategy='gap'` automatically calculates the largest probability drop to figure out exactly how many labels a sample deserves, eliminating the need for hyperparameter tuning.

We benchmark it against Sklearn's `OneVsRestClassifier` on the real-world **OpenML Scene Dataset**.

In [ ]:
import time
import pandas as pd
import torch
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from IPython.display import display

from fast_kmeans import FastMultiLabelKMeansClassifier

# 1. Load Multi-Label Dataset (Scene Images)
print("Downloading Scene Image Dataset from OpenML...")
X_np, y_np = fetch_openml('scene', version=1, return_X_y=True, as_frame=False)
y_matrix = np.array([[float(val) for val in row] for row in y_np], dtype=np.float32)
X_scaled = StandardScaler().fit_transform(X_np.astype(np.float32))

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_matrix, test_size=0.2, random_state=42)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

print("🚀 Benchmarking FastMultiLabelKMeans vs Sklearn OneVsRest(LogisticRegression)...\n")

# --- SKLEARN ONE-VS-REST ---
start = time.time()
ovr = OneVsRestClassifier(LogisticRegression(max_iter=500), n_jobs=-1)
ovr.fit(X_train, y_train)
ovr_preds = ovr.predict(X_test)
ovr_time = time.time() - start
ovr_f1 = f1_score(y_test, ovr_preds, average='macro')

# --- FAST-KMEANS MULTI-LABEL ---
start = time.time()
fkm_ml = FastMultiLabelKMeansClassifier(
    k_init='auto', 
    distance='cosine',
    asl_gamma_neg=2.0, # Asymmetric focal decay for easy negatives
    asl_gamma_pos=1.0, 
    repulsion_factor=1,
    negative_sampling=2,
)
fkm_ml.fit(X_train_t, y_train_t, max_iters=1000, verbose=True)
fkm_ml.finetune(X_train_t, y_train_t, epochs=100, early_stopping_rounds=10,
                eval_fraction=0.15, lr=1e-2, verbose=True)

# Gap strategy dynamically decides how many tags to apply
fkm_preds = fkm_ml.predict(X_test_t, strategy='gap').cpu().numpy()
fkm_time = time.time() - start
fkm_f1 = f1_score(y_test, fkm_preds, average='macro')

# 2. Build Comparison Table
results = pd.DataFrame({
    "Algorithm": ["Sklearn OneVsRest (LogReg)", "FastMultiLabelKMeans (ASL)"],
    "Architecture": ["Trains C independent models", "Independent Sub-topologies (Vectorized)"],
    "Imbalance Defense": ["None", "Asymmetric Loss (ASL) dynamic penalty"],
    "Macro F1-Score": [f"{ovr_f1:.4f}", f"{fkm_f1:.4f}"],
    "Total Time (Train+Predict)": [f"{ovr_time:.2f}s", f"{fkm_time:.2f}s"]
})

display(results.style.set_caption("Multi-Label Performance (OpenML Scene Dataset)")
        .set_properties(**{'text-align': 'center'}))

🚀 Benchmarking FastMultiLabelKMeans vs Sklearn OneVsRest(LogisticRegression)...



Fit (EMA Topology):   0%|          | 0/5000 [00:00<?, ?it/s]

Finetune (Gradient):   0%|          | 0/100 [00:00<?, ?it/s]

,Algorithm,Architecture,Imbalance Defense,Macro F1-Score,Total Time (Train+Predict)
0,Sklearn OneVsRest (LogReg),Trains C independent models,None,0.9617,6.56s
1,FastMultiLabelKMeans (ASL),Independent Sub-topologies (Vectorized),Asymmetric Loss (ASL) dynamic penalty,0.5933,45.14s
